# Predicting Loans - Data Cleaning

In this project we will model a borrower's credit risk. Credit has played a key role in the economy for centuries and some form of credit has existed since the beginning of commerce. We'll be working with financial lending data from Lending Club. Lending Club is a marketplace for personal loans that matches borrowers who are seeking a loan with investors looking to lend money and make a return. 

The dataset we use can be downloaded off of their website (https://www.lendingclub.com/auth/login?login_url=%2Fstatistics%2Fadditional-statistics%3F). For this project, we will focus on approved loans data from **2007 to 2011**, since a good number of these loans have already finished. In the datasets for later years, many of the loans are current and still being paid off.

The question we aim to anser is: 

**Can we build a machine learning model that can accurately predict if a borrower will pay off their loan on time or not?**

In [1]:
## Read in libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Prevent warning messages
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
## Read in data file
df = pd.read_csv('loans_2007.csv')

In [3]:
df.shape

(42538, 52)

In [4]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599.0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,...,171.62,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,1077430,1314167.0,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,...,119.66,Sep-2013,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,1077175,1313524.0,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,...,649.91,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
3,1076863,1277178.0,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,...,357.48,Apr-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
4,1075358,1311748.0,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,...,67.79,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0


## Note

We need to pay attention to any features that:

- leak information from the future (after the loan has already been funded)
- don't affect a borrower's ability to pay back a loan (e.g. a randomly generated ID value by Lending Club)
- are formatted poorly and need to be cleaned up
- require more data or a lot of processing to turn into a useful feature
- contain redundant information

We need to especially pay attention to data leakage, since it can cause our model to overfit. This is because the model would be using data about the target column that wouldn't be available when we're using the model on future loans.

### Drop Columns (1st Wave)

At first glance, we can conclude that the following features should be removed:

- ```id```: randomly generated field by Lending Club for unique identification purposes only
- ```member_id```: also a randomly generated field by Lending Club for unique identification purposes only
- ```funded_amnt```: leaks data from the future (after the loan is already started to be funded)
- ```funded_amnt_inv```: also leaks data from the future (after the loan is already started to be funded)
- ```grade```: contains redundant information from the interest rate column (```int_rate```)
- ```sub_grade```: also contains redundant information from the interest rate column (```int_rate```)
- ```emp_title```: requires other data and a lot of processing to potentially be useful
- ```issue_d```: leaks data from the future (after the loan is already completely funded)

In [8]:
df = df.drop(['id','member_id','funded_amnt','funded_amnt_inv','grade','sub_grade','emp_title','issue_d'],axis=1)

### Drop Columns (2nd Wave)

Within this group of columns, we need to drop the following columns:

- ```zip_code```: redundant with the addr_state column since only the first 3 digits of the 5 digit zip code are visible (which only can be used to identify the state the borrower lives in)
- ```out_prncp```: leaks data from the future, (after the loan already started to be paid off)
- ```out_prncp_inv```: also leaks data from the future, (after the loan already started to be paid off)
- ```total_pymnt```: also leaks data from the future, (after the loan already started to be paid off)
- ```total_pymnt_inv```: also leaks data from the future, (after the loan already started to be paid off)
- ```total_rec_prncp```: also leaks data from the future, (after the loan already started to be paid off)

The out_prncp and out_prncp_inv both describe the outstanding principal amount for a loan, which is the remaining amount the borrower still owes. These 2 columns as well as the total_pymnt column describe properties of the loan after it's fully funded and started to be paid off. This information isn't available to an investor before the loan is fully funded and we don't want to include it in our model.

In [10]:
df = df.drop(['zip_code','out_prncp','out_prncp_inv','total_pymnt','total_pymnt_inv','total_rec_prncp'],axis=1)

### Drop Columns (Final Wave)

From the remaining fields, we will drop:

- ```total_rec_int```: leaks data from the future, (after the loan already started to be paid off),
- ```total_rec_late_fee```: also leaks data from the future, (after the loan already started to be paid off),
- ```recoveries```: also leaks data from the future, (after the loan already started to be paid off),
- ```collection_recovery_fee```: also leaks data from the future, (after the loan already started to be paid off),
- ```last_pymnt_d```: also leaks data from the future, (after the loan already started to be paid off),
- ```last_pymnt_amnt```: also leaks data from the future, (after the loan already started to be paid off).

In [13]:
df = df.drop(['total_rec_int','total_rec_late_fee','recoveries','collection_recovery_fee',
              'last_pymnt_d','last_pymnt_amnt'],axis=1)

### Choosing Response Variable

We will use ```loan_status``` as our dependent variable becuase it directly describes if a loan was paid off on time, delayed or defaulted on.

In [18]:
df['loan_status'].value_counts()

Fully Paid                                             33136
Charged Off                                             5634
Does not meet the credit policy. Status:Fully Paid      1988
Current                                                  961
Does not meet the credit policy. Status:Charged Off      761
Late (31-120 days)                                        24
In Grace Period                                           20
Late (16-30 days)                                          8
Default                                                    3
Name: loan_status, dtype: int64

From the investor's perspective, we're interested in trying to predict which loans will be paid off on time and which ones won't be. Only the ```Fully Paid``` and ```Charged Off``` values describe the final outcome of the loan. The other values describe loans that are still ongoing and where the jury is still out on if the borrower will pay back the loan on time or not. While the ```Default``` status resembles the ```Charged Off``` status, in Lending Club's eyes, loans that are charged off have essentially no chance of being repaid while default ones have a small chance.

We will focus only on rows with ```Fully Paid``` and ```Charged Off``` since these are the two values we are trying to predict.

### Note

We need to be weary of **class imbalance** between the positive and negative cases. While there are 33,136 loans that have been fully paid off, there are only 5,634 that were charged off. This class imbalance is a common problem in binary classification and during training, the model ends up having a strong bias towards predicting the class with more observations in the training set and will rarely predict the class with less observations. The stronger the imbalance, the more biased the model becomes.

In [23]:
## Drop unwanted values under loan_status

df = df[(df['loan_status']=='Fully Paid') | (df['loan_status']=='Charged Off')]
df['loan_status'].value_counts()

Fully Paid     33136
Charged Off     5634
Name: loan_status, dtype: int64

In [26]:
## Assign binary classifier to loan_status

mapping = {
    'loan_status': {
        'Fully Paid': 1,
        'Charged Off': 0,
    }
}

df = df.replace(mapping)

df['loan_status'].value_counts()

1    33136
0     5634
Name: loan_status, dtype: int64

### Unique Value Columns

Finally we will drop columns that have only 1 unique value as these do not add any information to each loan application. 

In [ ]:
orig_columns = df.columns
drop_columns = []
for col in orig_columns:
    col_series = df[col].dropna().unique()
    if len(col_series) == 1:
        drop_columns.append(col)

df = df.drop(drop_columns, axis=1)

**Export for New Notebook**

In [56]:
df.to_csv('loans_clean.csv',index=False)